# Titanic Survival Prediction

## Mini Project — Classification with Scikit-learn

Project ini bertujuan untuk membangun model machine learning untuk memprediksi apakah seorang penumpang Titanic **selamat (`Survived = 1`) atau tidak selamat (`Survived = 0`)** berdasarkan karakteristik penumpang.

### Tahapan Project
1. Data Loading & Initial Exploration
2. Data Cleaning
3. Feature Engineering
4. Data Preprocessing
5. Baseline Model Comparison
6. Hyperparameter Tuning
7. Tuned Model Evaluation
8. Final Model Comparison

### Models
- Logistic Regression
- Decision Tree
- Random Forest
- K-Nearest Neighbors (KNN)
- Support Vector Machine (SVM)

Evaluasi dilakukan menggunakan **5-Fold Stratified Cross-Validation**.

## 1. Import Libraries

In [1]:
# Data manipulation
import pandas as pd
import numpy as np

# Data visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Preprocessing & Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# Cross-validation & Hyperparameter Tuning
from sklearn.model_selection import (
    StratifiedKFold,
    cross_validate,
    GridSearchCV
)

# Machine Learning Models
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC

# Evaluation
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay,
    RocCurveDisplay
)

# Model Interpretation
from sklearn.inspection import permutation_importance

## 2. Load Dataset

Dataset yang digunakan adalah `Titanic-Dataset.csv`.

Pada tahap awal dilakukan pemeriksaan terhadap struktur dataset, tipe data, statistik deskriptif, dan missing values.

In [2]:
df = pd.read_csv("data/raw/Titanic-Dataset.csv")

df.head()
print(df.info())
print(df.describe())
df.isnull().sum()

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    str    
 4   Sex          891 non-null    str    
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    str    
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    str    
 11  Embarked     889 non-null    str    
dtypes: float64(2), int64(5), str(5)
memory usage: 83.7 KB
None
       PassengerId    Survived      Pclass         Age       SibSp  \
count   891.000000  891.000000  891.000000  714.000000  891.000000   
mean    446.000000    0.383838    2.308642   29.699118    0.523008   
std     257.353842    0.486592    0.836071  

PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64

## 3. Data Cleaning

### Handling Missing Values in `Embarked`

Kolom `Embarked` hanya memiliki 2 missing values dari 891 observasi. Karena jumlahnya sangat kecil, dua observasi tersebut dihapus.

In [3]:
df = df.dropna(subset=["Embarked"]).copy()

df.isnull().sum()

PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         0
dtype: int64

## 4. Feature Engineering

### 4.1 Extracting Passenger Title

Title diekstraksi dari `Name` karena dapat merepresentasikan karakteristik seperti gender, status sosial, dan kelompok usia.

In [4]:
df["Title"] = df["Name"].str.extract(r",\s*([^.]+)\.")

df["Title"].value_counts()

Title
Mr              517
Miss            181
Mrs             124
Master           40
Dr                7
Rev               6
Major             2
Mlle              2
Col               2
Don               1
Mme               1
Ms                1
Lady              1
Sir               1
Capt              1
the Countess      1
Jonkheer          1
Name: count, dtype: int64

### Grouping Rare Titles

Title dengan frekuensi rendah digabungkan menjadi kategori `Rare` untuk mengurangi cardinality.

In [5]:
rare_titles = [
    "Capt", "Col", "Don", "Dr", "Jonkheer",
    "Lady", "Major", "Mlle", "Mme", "Ms",
    "Rev", "Sir", "the Countess"
]

df["Title"] = df["Title"].replace(rare_titles, "Rare")

df["Title"].value_counts()

Title
Mr        517
Miss      181
Mrs       124
Master     40
Rare       27
Name: count, dtype: int64

### 4.2 Handling Missing `Age`

Missing `Age` diimputasi berdasarkan median dari kombinasi **`Title × Pclass`**. Pendekatan ini lebih informatif dibandingkan median global karena kedua fitur tersebut berhubungan dengan karakteristik penumpang.

In [6]:
age_median = df.groupby(
    ["Title", "Pclass"]
)["Age"].transform("median")

df["Age"] = df["Age"].fillna(age_median)

df["Age"].isnull().sum()

np.int64(0)

### 4.3 Handling `Cabin`

Nomor cabin diubah menjadi fitur `Deck`, yaitu huruf pertama dari nomor cabin.

Selain itu dibuat `CabinMissing`:
- `0` = cabin tersedia
- `1` = cabin tidak tersedia

Deck `G` dan `T` digabungkan menjadi `Rare`, sedangkan missing deck diberi label `Unknown`.

In [7]:
df["Deck"] = df["Cabin"].str[0]

rare_decks = ["G", "T"]
df["Deck"] = df["Deck"].replace(rare_decks, "Rare")

df["CabinMissing"] = df["Cabin"].isna().astype(int)

df.drop(columns=["Cabin"], inplace=True)

df["Deck"] = df["Deck"].fillna("Unknown")

df["Deck"].value_counts()

Deck
Unknown    687
C           59
B           45
D           33
E           32
A           15
F           13
Rare         5
Name: count, dtype: int64

### 4.4 Family Features

Dibuat dua fitur baru:

$$FamilySize = SibSp + Parch + 1$$

`IsAlone` bernilai 1 jika penumpang bepergian sendiri.

In [8]:
df["FamilySize"] = (
    df["SibSp"] +
    df["Parch"] +
    1
)

df["IsAlone"] = (
    df["FamilySize"] == 1
).astype(int)

### 4.5 Fare per Person

Untuk memperhitungkan ukuran keluarga, dibuat:

$$FarePerPerson = \frac{Fare}{FamilySize}$$

Quartile `FarePerPerson` digunakan untuk exploratory analysis.

In [9]:
df["FarePerPerson"] = (
    df["Fare"] / df["FamilySize"]
)

fare_group = pd.qcut(
    df["FarePerPerson"],
    q=4,
    duplicates="drop"
)

df.groupby(
    fare_group,
    observed=True
)["Survived"].agg(
    ["count", "mean"]
).round(2)

,count,mean
FarePerPerson,,
"(-0.001, 7.25]",226,0.27
"(7.25, 8.158]",219,0.26
"(8.158, 22.525]",222,0.41
"(22.525, 512.329]",222,0.60


## 5. Final Data Check

Sebelum modeling, dilakukan pengecekan struktur data, missing values, dan distribusi target.

In [10]:
print(df.info())

print("\nMissing Values:")
print(df.isnull().sum())

print(df["Survived"].value_counts())

print(
    df["Survived"]
    .value_counts(normalize=True)
    .round(3)
)

<class 'pandas.DataFrame'>
Index: 889 entries, 0 to 890
Data columns (total 17 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   PassengerId    889 non-null    int64  
 1   Survived       889 non-null    int64  
 2   Pclass         889 non-null    int64  
 3   Name           889 non-null    str    
 4   Sex            889 non-null    str    
 5   Age            889 non-null    float64
 6   SibSp          889 non-null    int64  
 7   Parch          889 non-null    int64  
 8   Ticket         889 non-null    str    
 9   Fare           889 non-null    float64
 10  Embarked       889 non-null    str    
 11  Title          889 non-null    str    
 12  Deck           889 non-null    str    
 13  CabinMissing   889 non-null    int64  
 14  FamilySize     889 non-null    int64  
 15  IsAlone        889 non-null    int64  
 16  FarePerPerson  889 non-null    float64
dtypes: float64(3), int64(8), str(6)
memory usage: 125.0 KB
None

Missing V

## 6. Define Features & Target

`Survived` menjadi target variable.

Kolom `PassengerId`, `Name`, dan `Ticket` tidak digunakan langsung sebagai predictor. Informasi `Name` sudah direpresentasikan melalui `Title`.

In [11]:
X = df.drop(
    columns=[
        "Survived",
        "PassengerId",
        "Name",
        "Ticket"
    ]
)

y = df["Survived"]

numeric_features = [
    "Age",
    "SibSp",
    "Parch",
    "Fare",
    "FamilySize",
    "FarePerPerson"
]

categorical_features = [
    "Pclass",
    "Sex",
    "Embarked",
    "Title",
    "Deck",
    "CabinMissing",
    "IsAlone"
]

## 7. Data Preprocessing

Preprocessing dibangun menggunakan `ColumnTransformer` dan `Pipeline` agar diterapkan secara konsisten pada setiap fold cross-validation.

In [12]:
numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

## 8. Cross-Validation Strategy

Digunakan **5-Fold Stratified Cross-Validation** dengan `shuffle=True` dan `random_state=42`.

`StratifiedKFold` menjaga proporsi kelas target pada setiap fold.

In [13]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc"
}

### Evaluation Metrics

| Metric | Tujuan |
|---|---|
| Accuracy | Proporsi prediksi yang benar |
| Precision | Ketepatan prediksi kelas positif |
| Recall | Kemampuan menemukan kelas positif |
| F1-score | Keseimbangan precision dan recall |
| ROC-AUC | Kemampuan membedakan kelas positif dan negatif |

**F1-score** digunakan sebagai metric utama untuk hyperparameter tuning.

## 9. Baseline Modeling

Lima algoritma digunakan dengan konfigurasi baseline/default.

In [14]:
models = {
    "Logistic Regression": LogisticRegression(
        max_iter=1000,
        random_state=42
    ),

    "Decision Tree": DecisionTreeClassifier(
        random_state=42
    ),

    "Random Forest": RandomForestClassifier(
        random_state=42
    ),

    "KNN": KNeighborsClassifier(),

    "SVM": SVC(
        probability=False,
        random_state=42
    )
}

pipelines = {
    name: Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", model)
        ]
    )
    for name, model in models.items()
}

In [15]:
baseline_results = []

for name, pipeline in pipelines.items():
    cv_result = cross_validate(
        pipeline,
        X,
        y,
        cv=cv,
        scoring=scoring,
        n_jobs=-1
    )

    baseline_results.append({
        "Stage": "Baseline",
        "Model": name,
        "Accuracy": cv_result["test_accuracy"].mean(),
        "Precision": cv_result["test_precision"].mean(),
        "Recall": cv_result["test_recall"].mean(),
        "F1": cv_result["test_f1"].mean(),
        "ROC-AUC": cv_result["test_roc_auc"].mean()
    })

baseline_results_df = pd.DataFrame(baseline_results)

baseline_results_df = (
    baseline_results_df
    .sort_values("F1", ascending=False)
    .reset_index(drop=True)
)

baseline_results_df.round(4)

,Stage,Model,Accuracy,Precision,Recall,F1,ROC-AUC
0,Baseline,Logistic Regression,0.8335,0.7908,0.7676,0.7787,0.8694
1,Baseline,SVM,0.8268,0.7978,0.7324,0.7627,0.8605
2,Baseline,Random Forest,0.8167,0.7780,0.7265,0.7505,0.8680
3,Baseline,KNN,0.8155,0.7834,0.7176,0.7484,0.8563
4,Baseline,Decision Tree,0.7964,0.7447,0.7118,0.7270,0.7786


## 10. Hyperparameter Tuning

Tiga model dipilih untuk tuning:
- Logistic Regression
- SVM
- Random Forest

`GridSearchCV` menggunakan 5-fold cross-validation dengan scoring `F1`.

### 10.1 Logistic Regression

In [16]:
logistic_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", LogisticRegression(
            max_iter=2000,
            random_state=42
        ))
    ]
)

logistic_param_grid = {
    "model__C": [0.01, 0.1, 1, 10, 100],
    "model__class_weight": [None, "balanced"],
    "model__solver": ["liblinear", "lbfgs"]
}

logistic_grid = GridSearchCV(
    estimator=logistic_pipeline,
    param_grid=logistic_param_grid,
    cv=cv,
    scoring="f1",
    n_jobs=-1,
    return_train_score=True
)

logistic_grid.fit(X, y)

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...m_state=42))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'model__C': [0.01, 0.1, ...], 'model__class_weight': [None, 'balanced'], 'model__solver': ['liblinear', 'lbfgs']}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'f1'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",StratifiedKFo... shuffle=True)
,"return_train_score return_train_score: bool, default=FalseIf ``False``, the ``cv_results_`` attribute will not include trainingscores.Computing training scores is used to get insights on how differentparameter settings impact the overfitting/underfitting trade-off.However computing the scores on the training set can be computationallyexpensive and is not strictly required to select the parameters thatyield the best generalization performance... versionadded:: 0.19.. versionchanged:: 0.21 Default value was changed from ``True`` to ``False``",True
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be availa

### 10.2 Support Vector Machine

In [17]:
svm_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", SVC(
            probability=False,
            random_state=42
        ))
    ]
)

svm_param_grid = {
    "model__C": [0.1, 1, 10, 100],
    "model__kernel": ["linear", "rbf"],
    "model__gamma": ["scale", "auto"]
}

svm_grid = GridSearchCV(
    estimator=svm_pipeline,
    param_grid=svm_param_grid,
    cv=cv,
    scoring="f1",
    n_jobs=-1,
    verbose=2,
    return_train_score=True
)

svm_grid.fit(X, y)

Fitting 5 folds for each of 16 candidates, totalling 80 fits


c:\Users\S4N0K\OneDrive\Desktop\Titanic Survival DS Mini Project\venv\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...m_state=42))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'model__C': [0.1, 1, ...], 'model__gamma': ['scale', 'auto'], 'model__kernel': ['linear', 'rbf']}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'f1'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",StratifiedKFo... shuffle=True)
,"verbose verbose: int, default=0Controls the verbosity of information printed during fitting, with highervalues yielding more detailed logging.- 0 : no messages are printed;- >=1 : summary of the total number of fits;- >=2 : computation time for each fold and parameter candidate;- >=3 : fold indices and scores;- >=10 : parameter candidate indices and START messages before each fit.",2
,"return_train_score return_train_score: bool, default=FalseIf ``False``, the ``cv_results_`` attribute will not include trainingscores.Computing training scores is used to get insights on how differentparameter settings impact the overfitting/underfitting trade-off.However computing the scores on the training set can be computationallyexpensive and is not strictly required to select the parameters thatyield the best generalization performance... versionadded:: 0.19.. versionchanged:: 0.21 Default value was changed from ``True`` to ``False``",True
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be se

### 10.3 Random Forest

In [18]:
rf_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", RandomForestClassifier(
            random_state=42,
            n_jobs=-1
        ))
    ]
)

rf_param_grid = {
    "model__n_estimators": [100, 200],
    "model__max_depth": [None, 10],
    "model__min_samples_split": [2, 5],
    "model__min_samples_leaf": [1, 2],
    "model__max_features": ["sqrt"]
}

rf_grid = GridSearchCV(
    estimator=rf_pipeline,
    param_grid=rf_param_grid,
    cv=cv,
    scoring="f1",
    n_jobs=-1,
    verbose=2,
    return_train_score=True
)

rf_grid.fit(X, y)

Fitting 5 folds for each of 16 candidates, totalling 80 fits


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...m_state=42))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'model__max_depth': [None, 10], 'model__max_features': ['sqrt'], 'model__min_samples_leaf': [1, 2], 'model__min_samples_split': [2, 5], ...}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'f1'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",StratifiedKFo... shuffle=True)
,"verbose verbose: int, default=0Controls the verbosity of information printed during fitting, with highervalues yielding more detailed logging.- 0 : no messages are printed;- >=1 : summary of the total number of fits;- >=2 : computation time for each fold and parameter candidate;- >=3 : fold indices and scores;- >=10 : parameter candidate indices and START messages before each fit.",2
,"return_train_score return_train_score: bool, default=FalseIf ``False``, the ``cv_results_`` attribute will not include trainingscores.Computing training scores is used to get insights on how differentparameter settings impact the overfitting/underfitting trade-off.However computing the scores on the training set can be computationallyexpensive and is not strictly required to select the parameters thatyield the best generalization performance... versionadded:: 0.19.. versionchanged:: 0.21 Default value was changed from ``True`` to ``False``",True
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_e

## 11. Best Hyperparameters

In [19]:
grid_models = {
    "Logistic Regression": logistic_grid,
    "SVM": svm_grid,
    "Random Forest": rf_grid
}

for name, grid in grid_models.items():
    print(f"\n{name}")
    print("-" * len(name))
    print("Best Parameters:")
    print(grid.best_params_)
    print(f"Best F1: {grid.best_score_:.4f}")


Logistic Regression
-------------------
Best Parameters:
{'model__C': 10, 'model__class_weight': None, 'model__solver': 'liblinear'}
Best F1: 0.7798

SVM
---
Best Parameters:
{'model__C': 1, 'model__gamma': 'auto', 'model__kernel': 'rbf'}
Best F1: 0.7686

Random Forest
-------------
Best Parameters:
{'model__max_depth': 10, 'model__max_features': 'sqrt', 'model__min_samples_leaf': 1, 'model__min_samples_split': 5, 'model__n_estimators': 200}
Best F1: 0.7633


## 12. Tuned Model Evaluation

In [20]:
def evaluate_cv(model, X, y, cv, scoring):
    results = cross_validate(
        model,
        X,
        y,
        cv=cv,
        scoring=scoring,
        n_jobs=-1
    )

    return {
        "Accuracy": results["test_accuracy"].mean(),
        "Precision": results["test_precision"].mean(),
        "Recall": results["test_recall"].mean(),
        "F1": results["test_f1"].mean(),
        "ROC-AUC": results["test_roc_auc"].mean()
    }

tuned_models = {
    "Logistic Regression Tuned":
        logistic_grid.best_estimator_,
    "SVM Tuned":
        svm_grid.best_estimator_,
    "Random Forest Tuned":
        rf_grid.best_estimator_
}

tuned_results = []

for name, model in tuned_models.items():
    result = evaluate_cv(
        model,
        X,
        y,
        cv,
        scoring
    )

    result["Stage"] = "Tuned"
    result["Model"] = name

    tuned_results.append(result)

tuned_results_df = pd.DataFrame(tuned_results)

tuned_results_df = tuned_results_df[
    [
        "Stage",
        "Model",
        "Accuracy",
        "Precision",
        "Recall",
        "F1",
        "ROC-AUC"
    ]
]

tuned_results_df.sort_values(
    "F1",
    ascending=False
).reset_index(drop=True).round(4)

,Stage,Model,Accuracy,Precision,Recall,F1,ROC-AUC
0,Tuned,Logistic Regression Tuned,0.8346,0.7931,0.7676,0.7798,0.8679
1,Tuned,SVM Tuned,0.8324,0.8109,0.7324,0.7686,0.8524
2,Tuned,Random Forest Tuned,0.8290,0.8063,0.7265,0.7633,0.8763


## 13. Final Model Comparison

Baseline dan tuned models digabungkan untuk melihat dampak hyperparameter tuning terhadap performa.

In [21]:
comparison_df = pd.concat(
    [
        baseline_results_df,
        tuned_results_df
    ],
    ignore_index=True
)

comparison_df = (
    comparison_df
    .sort_values("F1", ascending=False)
    .reset_index(drop=True)
)

comparison_df.round(4)

,Stage,Model,Accuracy,Precision,Recall,F1,ROC-AUC
0,Tuned,Logistic Regression Tuned,0.8346,0.7931,0.7676,0.7798,0.8679
1,Baseline,Logistic Regression,0.8335,0.7908,0.7676,0.7787,0.8694
2,Tuned,SVM Tuned,0.8324,0.8109,0.7324,0.7686,0.8524
3,Tuned,Random Forest Tuned,0.8290,0.8063,0.7265,0.7633,0.8763
4,Baseline,SVM,0.8268,0.7978,0.7324,0.7627,0.8605
5,Baseline,Random Forest,0.8167,0.7780,0.7265,0.7505,0.8680
6,Baseline,KNN,0.8155,0.7834,0.7176,0.7484,0.8563
7,Baseline,Decision Tree,0.7964,0.7447,0.7118,0.7270,0.7786


## 14. Result Summary

Berdasarkan hasil evaluasi, **Logistic Regression Tuned** menjadi kandidat model terbaik berdasarkan F1-score pada eksperimen ini.

Hyperparameter tuning memberikan peningkatan yang relatif kecil dibandingkan baseline Logistic Regression. Hal ini menunjukkan bahwa preprocessing dan feature engineering yang digunakan sudah cukup efektif.

### Next Steps
1. Confusion Matrix
2. ROC Curve
3. Feature / Model Interpretation
4. Error Analysis
5. Final Model Selection